In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.1 Tensors and shapes

> 🎯 **Goal:** Learn what a tensor is and how to read its three defining properties (values, shape, dtype) so shape errors stop being scary.

**Why this matters.** Every single thing inside a language model (the text you feed it, the numbers it learns, the predictions it makes) lives inside tensors. Before we can multiply, normalize, or train anything, we need to be fluent in this one data structure. Think of this lesson as learning the alphabet before writing sentences.

**The intuition.** A tensor is just a box of numbers with a known shape, like an egg carton. A single egg is a 0-D tensor (a scalar). A row of eggs is a 1-D tensor (a vector). A full carton with rows and columns is a 2-D tensor (a matrix). Stack several cartons and you get 3-D, and so on. The numbers are the eggs; the shape tells you how the carton is laid out.

**The mechanics.** Three things describe any tensor, and you should glance at all three constantly:

- **Values:** the actual numbers stored inside.
- **Shape:** a tuple of sizes per dimension. A shape of `(2, 3)` means 2 rows and 3 columns, so 6 numbers total. `ndim` is just how many entries that tuple has (here, 2).
- **dtype** (short for "data type"): what kind of number each slot holds. `torch.int64` holds whole numbers (integers); `torch.float32` holds decimals (floats). This matters because integers are used to *index* things (like which word in the vocabulary) while floats carry the smoothly-adjustable quantities a model learns from.

In the code below, `torch.arange(6)` makes the numbers 0 through 5 in a row, and `.reshape(2, 3)` rearranges those same 6 numbers into a 2-by-3 grid (no values change, only the layout). Then `.float()` makes a copy where every value is a float instead of an integer.

In [ ]:
x = torch.arange(6).reshape(2, 3)
print(x)
print("shape:", tuple(x.shape), "dtype:", x.dtype, "ndim:", x.ndim)
f = x.float()
print("as float, dtype:", f.dtype)

**What just happened.** `x` printed as a 2-by-3 grid of the integers 0 to 5. Its shape is `(2, 3)`, its dtype is `torch.int64`, and its ndim is `2`. After calling `.float()`, the copy `f` holds the same numbers but with dtype `torch.float32`. Notice that `.float()` did not change `x` itself; in PyTorch, converting a dtype gives you a new tensor and leaves the original untouched. Integers index tokens; floats carry activations and gradients. Casting between them is always explicit, which is a feature: it stops you from accidentally mixing the two.

⚠️ **Common pitfalls**
- Confusing shape with the number of values. A shape of `(2, 3)` is not 5 numbers; it is 2 times 3, so 6. Always multiply the entries.
- Assuming the dtype is what you want. PyTorch defaults to `int64` for `arange` and `float32` for random floats. Feeding an integer tensor into a layer that expects floats is one of the most common beginner errors, and the error message can be cryptic.

🏋️ **Try it yourself.** Change `reshape(2, 3)` to `reshape(3, 2)` and rerun. The same 6 numbers now fill a 3-by-2 grid, so the shape prints `(3, 2)`. Then try `reshape(6)` (a flat row) and `reshape(1, 6)` (a single row inside a 2-D box) and watch how `ndim` changes even though the values never do.

In [ ]:
assert tuple(x.shape) == (2, 3)
assert x.dtype == torch.int64
assert f.dtype == torch.float32